# Brazilian E-Commerce Dashboard

**Dataset:** [Olist — Brazilian E-Commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)  
**Run order:** setup → load → EDA cells.

Place all Kaggle CSVs in **`data/`** next to this notebook. Override with env **`OLIST_DATA_DIR`** if needed.

## 1 - Setup: paths and `load_olist_csvs()`

**Local (Jupyter, VS Code, Cursor):** no mounting needed. Just make sure the nine Kaggle CSVs sit in `data/` next to this notebook.

**Google Colab:** upload the whole `data/` folder to the session, or mount Drive and set `os.environ["OLIST_DATA_DIR"] = "/content/.../data"`, then re-run the setup cell below.

In [ ]:
# =============================================================================
# Data: ./data/  (nine Olist CSVs from Kaggle)
# =============================================================================
%matplotlib inline

from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")


def resolve_data_dir() -> Path:
    """Locate the folder that contains Olist CSV files."""
    env = os.environ.get("OLIST_DATA_DIR", "").strip()
    if env:
        p = Path(env).expanduser().resolve()
        if not p.is_dir():
            raise FileNotFoundError(f"OLIST_DATA_DIR is not a directory: {p}")
        return p

    try:
        nb_path = globals().get("__vsc_ipynb_file__")
        if nb_path:
            cand = Path(nb_path).resolve().parent / "data"
            if cand.is_dir() and any(cand.glob("olist_*_dataset.csv")):
                return cand
    except Exception:
        pass

    start = Path.cwd().resolve()
    for folder in [start, *start.parents]:
        candidate = folder / "data"
        if candidate.is_dir() and any(candidate.glob("olist_*_dataset.csv")):
            return candidate
    for folder in [start, *start.parents]:
        candidate = folder / "data"
        if candidate.is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find `data/` with Olist CSVs. Put CSVs in ./data next to this notebook, "
        "or set environment variable OLIST_DATA_DIR."
    )


def load_olist_csvs(data_dir: Path) -> dict[str, pd.DataFrame]:
    """Load every *.csv under ``data_dir``. Keys = CSV stem names (sorted)."""
    data_dir = Path(data_dir).resolve()
    paths = sorted(data_dir.glob("*.csv"))
    if not paths:
        raise FileNotFoundError(f"No CSV files in {data_dir}")

    print(f"Reading from:\n  {data_dir}\nFiles ({len(paths)}):")
    for fp in paths:
        print(f"  - {fp.name}")

    tables: dict[str, pd.DataFrame] = {}
    for fp in paths:
        key = fp.stem
        try:
            tables[key] = pd.read_csv(fp, encoding="utf-8")
            print(f"  OK  {fp.name}  ->  {key}  {tables[key].shape}")
        except Exception as exc:
            print(f"  ERR {fp.name}  ->  {exc}")

    expected = {
        "olist_customers_dataset",
        "olist_geolocation_dataset",
        "olist_order_items_dataset",
        "olist_order_payments_dataset",
        "olist_order_reviews_dataset",
        "olist_orders_dataset",
        "olist_products_dataset",
        "olist_sellers_dataset",
        "product_category_name_translation",
    }
    miss = sorted(expected - set(tables))
    if miss:
        print("Warning — missing expected table(s):", ", ".join(miss))

    return tables


DATA_DIR = resolve_data_dir()
print("DATA_DIR =", DATA_DIR.resolve())


## 2 — Load every CSV → `all_dataframes`

In [ ]:
from IPython.display import display

assert "load_olist_csvs" in globals() and "DATA_DIR" in globals(), (
    "Run the setup cell above first."
)

all_dataframes = load_olist_csvs(DATA_DIR)

if all_dataframes:
    preview_key = sorted(all_dataframes.keys())[0]
    print(f"\nPreview ({preview_key}):")
    display(all_dataframes[preview_key].head())


## 3 — Quick overview per table

`olist_geolocation_dataset` has ~1M rows; `head()`/`info()` are still cheap.

In [ ]:
from IPython.display import display

for df_name, df in all_dataframes.items():
    print(f"\n--- {df_name} ---")
    print("Shape:", df.shape)
    display(df.head())
    df.info()
